<a href="https://colab.research.google.com/github/anastasiakalyashova/python-ai-AnastasiaKalyashova/blob/main/week3f_rock_cooccurrence_graph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ═══════════════════════════════════════════════════════
#  ЯЧЕЙКА 0. Подготовка данных (из week2b_read_csv.ipynb)
#  Запускать первой в каждом ноутбуке задания 3
# ═══════════════════════════════════════════════════════

# --- Параметры (изменять здесь) ----------------------
RADIUS_KM     = 300   # радиус соседства гор (для week3a)
TOP_N_ROCKS   = 10    # сколько топ-пород использовать
TOP_N_COMPLEX = 20    # сколько самых «сложных» гор брать
# -----------------------------------------------------

import os, pandas as pd, numpy as np
from itertools import combinations

# 1. Клонируем репозиторий (если ещё нет)
repo = "python-ai-AnastasiaKalyashova"
repo_path = f"/content/{repo}"
if not os.path.exists(repo_path):
    !git clone -q https://github.com/anastasiakalyashova/python-ai-AnastasiaKalyashova.git
if os.getcwd() != repo_path:
    %cd {repo_path}

# 2. Читаем CSV
file_path = None
for root, dirs, files in os.walk("."):
    if "mountains.csv" in files:
        file_path = os.path.join(root, "mountains.csv")
        break
df = pd.read_csv(file_path)

# 3. Переименование столбцов
if "mountainLabel" in df.columns:
    df = df.rename(columns={
        "mountain":          "URL",
        "mountainLabel":     "mountain",
        "rockMaterialLabel": "rockMaterial",
        "elevationMeters":   "elevation",
    })

# 4. Нормализуем породы
df["rockMaterial"] = df["rockMaterial"].str.lower().str.strip()

# 🔧 ИСПРАВЛЕНИЕ: заменяем "lutite" на "пелит"
df["rockMaterial"] = df["rockMaterial"].replace("lutite", "пелит")

# 5. Парсим координаты
coords = df["coordinates"].str.extract(r'Point\(([^\s]+)\s+([^\s]+)\)')
df["lon"] = pd.to_numeric(coords[0], errors="coerce")
df["lat"] = pd.to_numeric(coords[1], errors="coerce")

# 6. df_unique — по одной строке на гору
df_unique = (
    df.groupby("URL")
    .agg(
        mountain   = ("mountain",     "first"),
        lon        = ("lon",          "first"),
        lat        = ("lat",          "first"),
        elevation  = ("elevation",    "first"),
        rock_count = ("rockMaterial", "nunique"),
        rocks      = ("rockMaterial", lambda x: list(x.unique())),
    )
    .reset_index()
)

# 7. df_clean — только физически возможные высоты
df_clean = df_unique[
    (df_unique.elevation >= 0) &
    (df_unique.elevation <= 8849)
].copy()

# 8. Топ пород по частоте (по df_clean)
top_rocks = (
    df[df["URL"].isin(df_clean["URL"])]
    ["rockMaterial"].value_counts()
    .head(TOP_N_ROCKS).index.tolist()
)

# 9. Co-occurrence матрица пород
pairs = []
for rocks in df_clean["rocks"]:
    clean = [r for r in rocks if r in top_rocks]
    pairs += list(combinations(sorted(set(clean)), 2))
cooc = (pd.DataFrame(pairs, columns=["r1", "r2"])
        .value_counts()
        .reset_index(name="count"))

print(f"✅ Длинный формат:    {len(df)} строк")
print(f"✅ Уникальных гор:    {len(df_unique)}")
print(f"✅ df_clean:          {len(df_clean)} гор (0–8849 м)")
print(f"✅ Топ-{TOP_N_ROCKS} пород:    {top_rocks}")
print(f"✅ Пар co-occurrence: {len(cooc)}")

/content/python-ai-AnastasiaKalyashova
✅ Длинный формат:    4431 строк
✅ Уникальных гор:    2915
✅ df_clean:          2914 гор (0–8849 м)
✅ Топ-10 пород:    ['известняк', 'песчаник', 'гранит', 'мергель', 'конгломерат', 'пелит', 'доломит', 'андезит', 'осадочная горная порода', 'базальт']
✅ Пар co-occurrence: 15


In [7]:
# ═══════════════════════════════════════════════════════
# week3f_rock_cooccurrence_graph_v2.ipynb
# Альтернативная визуализация: КРУГОВОЙ КЛАСТЕРНЫЙ ГРАФ
# ═══════════════════════════════════════════════════════

import networkx as nx
import plotly.graph_objects as go
import numpy as np
import pandas as pd
from collections import Counter

# ─────────────────────────────────────────────────────
# 1. ГЕОЛОГИЧЕСКАЯ КЛАССИФИКАЦИЯ (та же)
# ─────────────────────────────────────────────────────

ROCK_FAMILIES = {
    'магматические': [
        'гранит', 'андезит', 'базальт', 'риолит', 'габбро', 'диорит',
        'гранодиорит', 'сиенит', 'дунит', 'перидотит', 'пикрит',
        'порфир', 'дацит', 'трахит', 'фонолит', 'туф', 'обсидиан',
        'пемза', 'лабрадорит', 'монцонит', 'тонит'
    ],
    'метаморфические': [
        'метаморфическая горная порода', 'гнейс', 'сланец', 'кварцит',
        'мрамор', 'амфиболит', 'серпентинит', 'эклогит', 'гранулит',
        'филлит', 'кристаллические сланцы', 'зеленокаменная порода',
        'роговик', 'милонит', 'катаклазит'
    ],
    'осадочные': [
        'известняк', 'песчаник', 'мергель', 'конгломерат', 'пелит',
        'доломит', 'аргиллит', 'алевролит', 'сланец глинистый',
        'осадочная горная порода', 'глина', 'ил', 'мел', 'трепел',
        'диатомит', 'эвапорит', 'гипс', 'ангидрит', 'каменная соль',
        'фосфорит', 'брекчия', 'флиш', 'моласса'
    ],
    'ледниковые': ['лёд', 'лед', 'льда', 'ледник', 'фирн', 'гляциал'],
    'другие': [
        'горная порода', 'вулканический бомб', 'пирокластическая порода',
        'карбонатная порода', 'кремнистая порода', 'глинозём'
    ]
}

rock_to_family = {}
for family, rocks in ROCK_FAMILIES.items():
    for rock in rocks:
        rock_to_family[rock] = family

FAMILY_COLORS = {
    'магматические': '#FF6B6B',
    'метаморфические': '#4ECDC4',
    'осадочные': '#45B7D1',
    'ледниковые': '#96CEB4',
    'другие': '#FFEAA7'
}

FAMILY_NAMES_RU = {
    'магматические': '🔥 Магматические',
    'метаморфические': '🪨 Метаморфические',
    'осадочные': '🏔️ Осадочные',
    'ледниковые': '❄️ Ледниковые',
    'другие': '📎 Другие'
}

# ─────────────────────────────────────────────────────
# 2. ПОСТРОЕНИЕ ГРАФА
# ─────────────────────────────────────────────────────

G = nx.Graph()
for rock in top_rocks:
    family = rock_to_family.get(rock, 'другие')
    G.add_node(rock, family=family)

for _, row in cooc.iterrows():
    r1, r2, count = row['r1'], row['r2'], row['count']
    if r1 in top_rocks and r2 in top_rocks:
        G.add_edge(r1, r2, weight=count)

rock_counts = df[df["URL"].isin(df_clean["URL"])]["rockMaterial"].value_counts()

# ─────────────────────────────────────────────────────
# 3. КРУГОВАЯ РАСКЛАДКА (circular layout)
# ─────────────────────────────────────────────────────

# Сортируем узлы по семьям для кластеризации
nodes_by_family = {}
for node in G.nodes():
    family = G.nodes[node]['family']
    if family not in nodes_by_family:
        nodes_by_family[family] = []
    nodes_by_family[family].append(node)

# Сортируем внутри семей по популярности
for family in nodes_by_family:
    nodes_by_family[family].sort(key=lambda x: rock_counts.get(x, 0), reverse=True)

# Создаём упорядоченный список узлов
ordered_nodes = []
for family in ['магматические', 'метаморфические', 'осадочные', 'ледниковые', 'другие']:
    if family in nodes_by_family:
        ordered_nodes.extend(nodes_by_family[family])

# Круговая раскладка
n_nodes = len(ordered_nodes)
angles = np.linspace(0, 2*np.pi, n_nodes, endpoint=False)
radius = 1.0

pos = {}
for i, node in enumerate(ordered_nodes):
    angle = angles[i]
    pos[node] = (radius * np.cos(angle), radius * np.sin(angle))

# ─────────────────────────────────────────────────────
# 4. ПОДГОТОВКА ДАННЫХ ДЛЯ ВИЗУАЛИЗАЦИИ
# ─────────────────────────────────────────────────────

# Рёбра с градиентом по весу
edge_x, edge_y, edge_colors, edge_widths = [], [], [], []
edge_weights = []

for u, v in G.edges():
    x0, y0 = pos[u]
    x1, y1 = pos[v]
    weight = G[u][v]['weight']
    edge_weights.append(weight)

    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])

    # Нормализуем цвет от светлого к тёмному
    edge_colors.extend(['rgba(100,100,100,0.3)'] * 3)

# Нормализация толщины рёбер
if edge_weights:
    max_w = max(edge_weights)
    min_w = min(edge_weights)
    edge_widths = [(w - min_w) / (max_w - min_w) * 6 + 1 for w in edge_weights] if max_w != min_w else [3]*len(edge_weights)

# Узлы
node_x, node_y, node_sizes, node_colors, node_texts = [], [], [], [], []

for node in ordered_nodes:
    x, y = pos[node]
    node_x.append(x)
    node_y.append(y)

    count = rock_counts.get(node, 0)
    family = G.nodes[node]['family']
    size = np.log1p(count) * 8 + 12

    node_sizes.append(size)
    node_colors.append(FAMILY_COLORS.get(family, '#DDDDDD'))
    node_texts.append(
        f"<b>{node}</b><br>"
        f"🏷️ Семья: {FAMILY_NAMES_RU.get(family, family)}<br>"
        f"📊 Встречаемость: {count} гор"
    )

# ─────────────────────────────────────────────────────
# 5. ВИЗУАЛИЗАЦИЯ
# ─────────────────────────────────────────────────────

fig = go.Figure()

# Добавляем рёбра (каждое ребро отдельно для разной толщины)
for i, (u, v) in enumerate(G.edges()):
    x0, y0 = pos[u]
    x1, y1 = pos[v]
    weight = G[u][v]['weight']
    width = edge_widths[i]

    fig.add_trace(go.Scatter(
        x=[x0, x1], y=[y0, y1],
        mode='lines',
        line=dict(width=width, color=f'rgba(100,100,100,{0.2 + width/10})'),
        hoverinfo='none',
        showlegend=False
    ))

# Добавляем узлы
fig.add_trace(go.Scatter(
    x=node_x, y=node_y,
    mode='markers+text',
    text=ordered_nodes,
    textposition='top center',
    textfont=dict(size=11, color='#333', family='Arial'),
    hoverinfo='text',
    hovertext=node_texts,
    marker=dict(
        size=node_sizes,
        color=node_colors,
        line=dict(width=2, color='white'),
        opacity=0.9
    ),
    showlegend=False
))

# ─────────────────────────────────────────────────────
# 6. ДИЗАЙН И ЛЕГЕНДА (ВНУТРИ, НО НЕ ПЕРЕКРЫВАЕТ)
# ─────────────────────────────────────────────────────

fig.update_layout(
    title=dict(
        text=f"🌍 <b>Геологическое родство пород</b><br>"
             f"<sup>Круговая кластеризация по семьям | Топ-{TOP_N_ROCKS} пород</sup>",
        x=0.5,
        font=dict(size=20)
    ),
    width=1100,
    height=900,
    margin=dict(l=100, r=100, t=120, b=80),
    xaxis=dict(
        visible=False,
        range=[-1.3, 1.3],
        showgrid=False,
        zeroline=False
    ),
    yaxis=dict(
        visible=False,
        range=[-1.3, 1.3],
        showgrid=False,
        zeroline=False,
        scaleanchor="x",
        scaleratio=1
    ),
    plot_bgcolor='rgba(250,250,250,0.9)',
    paper_bgcolor='white',
    hovermode='closest'
)

# Добавляем легенду в левый верхний угол (компактно)
legend_x, legend_y = -1.25, 1.15
for family, color in FAMILY_COLORS.items():
    fig.add_annotation(
        x=legend_x, y=legend_y,
        xref="x", yref="y",
        text=f"● {FAMILY_NAMES_RU.get(family, family)}",
        showarrow=False,
        font=dict(size=12, color=color),
        align="left",
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor=color,
        borderwidth=1,
        borderpad=4
    )
    legend_y -= 0.08

# Статистика в правом верхнем углу
stats_text = (
    f"<b>📊 Статистика</b><br>"
    f"Узлов: {G.number_of_nodes()}<br>"
    f"Рёбер: {G.number_of_edges()}<br>"
    f"Плотность: {nx.density(G):.3f}"
)

fig.add_annotation(
    x=1.25, y=1.15,
    xref="x", yref="y",
    text=stats_text,
    showarrow=False,
    font=dict(size=11),
    align="left",
    bgcolor="rgba(255,255,255,0.9)",
    bordercolor="#999",
    borderwidth=1,
    borderpad=8
)

# Добавляем информационную панель внизу
fig.add_annotation(
    x=0, y=-1.25,
    xref="x", yref="y",
    text="💡 <b>Совет:</b> Наведите курсор на узел для деталей | Толщина линии = сила связи",
    showarrow=False,
    font=dict(size=11, color="#666"),
    align="center",
    bgcolor="rgba(255,255,255,0.7)",
    bordercolor="#CCC",
    borderwidth=1,
    borderpad=5
)

fig.show()

# ─────────────────────────────────────────────────────
# 7. АНАЛИЗ И ВЫВОДЫ
# ─────────────────────────────────────────────────────

print("\n" + "="*80)
print("📊 КРУГОВОЙ КЛАСТЕРНЫЙ ГРАФ - РЕЗУЛЬТАТЫ")
print("="*80)

# Метрики кластеризации
print(f"\n🎯 Узлов: {G.number_of_nodes()}, Рёбер: {G.number_of_edges()}")
print(f"📈 Плотность графа: {nx.density(G):.3f}")

# Анализ связей внутри/между семьями
internal = 0
external = 0
for u, v in G.edges():
    if G.nodes[u]['family'] == G.nodes[v]['family']:
        internal += 1
    else:
        external += 1

print(f"\n🔗 Связи внутри семей: {internal} ({internal/G.number_of_edges()*100:.1f}%)")
print(f"🔗 Связи между семьями: {external} ({external/G.number_of_edges()*100:.1f}%)")

if internal > external:
    print("\n✅ ВЫВОД: Круговая кластеризация ОПРАВДАНА — породы одной семьи действительно связаны!")
else:
    print("\n⚠️ ВЫВОД: Преобладают межсемейные связи — геологические процессы сложнее.")

# Топ-3 центральных узла (по degree centrality)
centrality = nx.degree_centrality(G)
top_central = sorted(centrality.items(), key=lambda x: -x[1])[:3]
print("\n⭐ Самые центральные породы (хабы):")
for node, cent in top_central:
    family = G.nodes[node]['family']
    print(f"   • {node} ({FAMILY_NAMES_RU.get(family, family)}) — centrality: {cent:.3f}")

# Топ-3 самых сильных связей
print("\n🔗 Самые сильные геологические связи:")
strong_edges = sorted(G.edges(data=True), key=lambda x: -x[2]['weight'])[:3]
for u, v, data in strong_edges:
    print(f"   • {u} ⟷ {v} — встречаются вместе в {data['weight']} горах")

# Сохраняем
fig.write_html("rock_cooccurrence_graph_circular.html")
print("\n💾 Граф сохранён как 'rock_cooccurrence_graph_circular.html'")


📊 КРУГОВОЙ КЛАСТЕРНЫЙ ГРАФ - РЕЗУЛЬТАТЫ

🎯 Узлов: 10, Рёбер: 15
📈 Плотность графа: 0.333

🔗 Связи внутри семей: 13 (86.7%)
🔗 Связи между семьями: 2 (13.3%)

✅ ВЫВОД: Круговая кластеризация ОПРАВДАНА — породы одной семьи действительно связаны!

⭐ Самые центральные породы (хабы):
   • известняк (🏔️ Осадочные) — centrality: 0.667
   • мергель (🏔️ Осадочные) — centrality: 0.556
   • песчаник (🏔️ Осадочные) — centrality: 0.444

🔗 Самые сильные геологические связи:
   • известняк ⟷ мергель — встречаются вместе в 177 горах
   • песчаник ⟷ пелит — встречаются вместе в 170 горах
   • песчаник ⟷ конгломерат — встречаются вместе в 141 горах

💾 Граф сохранён как 'rock_cooccurrence_graph_circular.html'
